In [30]:
import asyncio
from playwright.async_api import async_playwright
import math
import random

BASE_URL = "https://www.roseltorg.ru/imuschestvo/nedvizhimost/kommercheskaya-nedvizhimost"
PARAMS = "sale=all&okato[]=45000000000&status[]=5&status[]=0&status[]=1"
ITEMS_PER_PAGE = 40

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/121 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/119 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/123 Safari/537.36"
]

async def get_total_count(page):
    ttl = page.locator("div.search-category__headtitle span")
    text = (await ttl.all_inner_texts())[0]
    return int(text.replace(" ", ""))

async def parse_page(browser, page_num, retries=3):
    for attempt in range(retries):
        #page = await browser.new_page()
        context = await browser.new_context(
            user_agent=random.choice(USER_AGENTS),
            viewport={"width": 1920, "height": 1080},
            locale="ru-RU",
        )
    
        page = await context.new_page()
        try:
            url = f"{BASE_URL}?{PARAMS}&page={page_num}"
            print(f"[Page {page_num}] Attempt {attempt}")

            await asyncio.sleep(random.uniform(2, 5))
            await page.goto(url, wait_until="domcontentloaded")

            await page.wait_for_selector(
                "div.search-item",
                timeout=10000   # shorter
            )

            # SUCCESS → extract
            data = []
            items = page.locator(
                "div.search-item:has-text('Начальная стоимость')"
            )

            for i in range(await items.count()):
                item = items.nth(i)

                info_block = item.locator(".search-item__information")
                texts = await info_block.all_inner_texts()

                data.append({
                    "page": page_num,
                    "info": texts[0] if texts else None
                })

            await page.close()
            print(f"[Page {page_num}] SUCCESS")
            await context.close()
            
            return data

        except Exception as e:
            print(f"[Page {page_num}] FAIL attempt {attempt}: {e}")
            await context.close()
            await page.close()

            await asyncio.sleep(2 + attempt * 2)

    print(f"[Page {page_num}] FAILED completely")
    return []


async def main():
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=["--disable-blink-features=AutomationControlled"]
        )
        context = await browser.new_context(
            viewport={"width": 1920, "height": 1080},
            user_agent=random.choice(USER_AGENTS),
            #user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120 Safari/537.36",
            locale="ru-RU",
        )
        
        await context.add_init_script("""
        Object.defineProperty(navigator, 'webdriver', {
            get: () => undefined
        })
        """)
        
        page = await context.new_page()
        
        await page.goto(f"{BASE_URL}?{PARAMS}&page=1", wait_until="domcontentloaded")
        #await page.wait_for_load_state("networkidle")
        
        html = await page.content()
        print("URL:", page.url)
        print(html[:1000])
        
        await page.wait_for_selector("div.search-item", timeout=60000)

        total = await get_total_count(page)
        total_pages = math.ceil(total / ITEMS_PER_PAGE)

        print("Total pages:", total_pages)

        semaphore = asyncio.Semaphore(2)
        
        async def parse_page_limited(browser, page_num):
            async with semaphore:
                return await parse_page(browser, page_num)
        
        #параллельные вычисления
        tasks = [
            parse_page_limited(browser, i)
            for i in range(1, total_pages + 1)
        ]

        results = await asyncio.gather(*tasks)

        await browser.close()

    #итог
    all_data = [item for sublist in results for item in sublist]

    print("Total collected:", len(all_data))
    return all_data


data = await main()


URL: https://www.roseltorg.ru/imuschestvo/nedvizhimost/kommercheskaya-nedvizhimost?sale=all&okato[]=45000000000&status[]=5&status[]=0&status[]=1&page=1
<!DOCTYPE html><html lang="ru" dir="ltr" prefix="content: http://purl.org/rss/1.0/modules/content/  dc: http://purl.org/dc/terms/  foaf: http://xmlns.com/foaf/0.1/  og: http://ogp.me/ns#  rdfs: http://www.w3.org/2000/01/rdf-schema#  schema: http://schema.org/  sioc: http://rdfs.org/sioc/ns#  sioct: http://rdfs.org/sioc/types#  skos: http://www.w3.org/2004/02/skos/core#  xsd: http://www.w3.org/2001/XMLSchema# " class=" js" style="--vw: 19.2px;"><head><style data-tippy-stylesheet="">.tippy-tooltip[data-animation=fade][data-state=hidden]{opacity:0}.tippy-iOS{cursor:pointer!important;-webkit-tap-highlight-color:transparent}.tippy-popper{pointer-events:none;max-width:calc(100vw - 10px);transition-timing-function:cubic-bezier(.165,.84,.44,1);transition-property:transform}.tippy-tooltip{position:relative;color:#fff;border-radius:4px;font-size:

In [31]:
import re

def parse_item(info_text):
    if not info_text:
        return {}

    lines = [l.strip() for l in info_text.split("\n") if l.strip()]

    result = {
        "title": None,
        "price": None,
        "address": None,
        "seller": None
    }

    # ----------------------
    # TITLE (usually first meaningful line)
    # ----------------------
    if lines:
        result["title"] = lines[0]

    # ----------------------
    # PRICE
    # ----------------------
    for i, line in enumerate(lines):
        if "Начальная стоимость" in line:
            if i + 1 < len(lines):
                result["price"] = lines[i + 1]
            break

    # ----------------------
    # SELLER
    # ----------------------
    for i, line in enumerate(lines):
        if "Продавец" in line:
            if i + 1 < len(lines):
                result["seller"] = lines[i + 1]
            break

    # ----------------------
    # ADDRESS (heuristic)
    # ----------------------
    for line in lines:
        if any(word in line for word in ["шоссе", "улица", "проспект", "дом"]):
            result["address"] = line
            break

    return result

import pandas as pd 

output = {}  
d = []
for i, row in enumerate(data):
    #print(f'{i}:{row}')
    #print(f'{i}:{row['page']}:{row['info']}')
    #print(parse_item(row['info']))
    #parse_item(row['info'])
    d.append(parse_item(row['info']))

df = pd.DataFrame(d)    
df.to_excel('parser_data.xlsx', index=False, sheet_name='property')

In [25]:
print(df)

                                                 title price  \
0    На право заключения договора аренды нежилыми п...  None   
1                         Нежилое помещение, 51,6 кв.м  None   
2                           Нежилое помещение, 92 кв.м  None   
3                         Нежилое помещение, 57,7 кв.м  None   
4                         Нежилое помещение, 24,4 кв.м  None   
..                                                 ...   ...   
385                   Коммерческое помещение, 3,9 кв.м  None   
386                     Коммерческое помещение, 2 кв.м  None   
387                 Коммерческое помещение, 145,2 кв.м  None   
388                  Коммерческое помещение, 12,3 кв.м  None   
389                  Коммерческое помещение, 11,9 кв.м  None   

                                               address  \
0    На право заключения договора аренды нежилыми п...   
1              улица Маршала Сергеева, дом 3, Этаж № 1   
2        Филёвский бульвар, дом 24, корпус 2, Этаж № 1   